# watersheds 12, 10, 8

### USGS National Map Watershed Boundary Dataset

In [20]:
from pathlib import Path
import requests
import time
import geopandas as gpd
import pyogrio
import pandas as pd
from shapely import make_valid
from tqdm.auto import tqdm
from shapely import make_valid
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
from osgeo import ogr
import shutil

In [5]:
import pyogrio
from pathlib import Path

# Correct watershed destination
watershed_gdb = Path(
    "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Watersheds_By_State/Alabama_Watersheds.gdb"
)

# Original gages geodatabase
gages_gdb = Path(
    "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Gages_State_GDBs(1)/Alabama.gdb"
)

# --------------------------------------------------
# 1. VERIFY HUC-12 IS IN THE WATERSHED GDB
# --------------------------------------------------

watershed_layers = [
    layer[0] for layer in pyogrio.list_layers(watershed_gdb)
]

if "HUC_12" not in watershed_layers:
    raise RuntimeError(
        "HUC_12 is not in Alabama_Watersheds.gdb. "
        "Gages GDB will not be modified."
    )

watershed_info = pyogrio.read_info(
    watershed_gdb,
    layer="HUC_12"
)

print("Watershed GDB verified:")
print("Features:", watershed_info["features"])
print("CRS:", watershed_info["crs"])

# --------------------------------------------------
# 2. REMOVE HUC-12 FROM THE GAGES GDB
# --------------------------------------------------

gages_layers = [
    layer[0] for layer in pyogrio.list_layers(gages_gdb)
]

if "HUC_12" in gages_layers:
    pyogrio.remove(gages_gdb, layer="HUC_12")
    print("\nRemoved HUC_12 from Alabama gages GDB.")
else:
    print("\nHUC_12 was not present in the gages GDB.")

# --------------------------------------------------
# 3. VERIFY BOTH GEODATABASES
# --------------------------------------------------

print("\nALABAMA WATERSHED GDB:")
print(pyogrio.list_layers(watershed_gdb))

print("\nALABAMA GAGES GDB:")
print(pyogrio.list_layers(gages_gdb))

Output directory:
/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Watersheds_By_State


PROCESSING: ALABAMA

Candidate watersheds: 67


HUC layer 4:   0%|          | 0/3 [00:00<?, ?it/s]

/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field objectid of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field loaddate of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(


Saved 53 watershed polygons.

Candidate watersheds: 440


HUC layer 5:   0%|          | 0/18 [00:00<?, ?it/s]

/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field objectid of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field loaddate of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(


... (82 outputs left out)

Saved 175 watershed polygons.

Candidate watersheds: 1,995


HUC layer 6:   0%|          | 0/80 [00:00<?, ?it/s]

/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field objectid of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field loaddate of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(


Saved 785 watershed polygons.

WATERSHED DOWNLOAD SUMMARY
        State HUC_Level  Feature_Count     Status       CRS
      Alabama     HUC_8             53 Downloaded EPSG:4269
      Alabama    HUC_10            321 Downloaded EPSG:4269
     Arkansas     HUC_8             58 Downloaded EPSG:4269
     Arkansas    HUC_10            308 Downloaded EPSG:4269
     Arkansas    HUC_12           1556 Downloaded EPSG:4269
         Iowa     HUC_8             56 Downloaded EPSG:4269
         Iowa    HUC_10            389 Downloaded EPSG:4269
         Iowa    HUC_12           1713 Downloaded EPSG:4269
     Kentucky     HUC_8             46 Downloaded EPSG:4269
     Kentucky    HUC_10            249 Downloaded EPSG:4269
     Kentucky    HUC_12           1302 Downloaded EPSG:4269
  Mississippi     HUC_8             55 Downloaded EPSG:4269
  Mississippi    HUC_10            327 Downloaded EPSG:4269
  Mississippi    HUC_12           1469 Downloaded EPSG:4269
     Missouri     HUC_8             67 Dow

### Downloaded gdbs with hcu 8, 10, and 12 for our 10 test states in a file called intermediate/

### Adding in alabama hcu 12 amending it to existing gdbs (errors)

In [16]:
gages_gdb = (
    "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/"
    "data/processed/Gages_State_GDBs(1)/Alabama.gdb"
)

driver = ogr.GetDriverByName("OpenFileGDB")
gdb = driver.Open(gages_gdb, 1)

if gdb is None:
    raise RuntimeError("Could not open the gages GDB for editing.")

layer = gdb.GetLayerByName("HUC_12")

if layer is not None:
    result = gdb.DeleteLayer("HUC_12")

    if result != 0:
        raise RuntimeError("GDAL could not delete the HUC_12 layer.")

    print("Successfully deleted HUC_12 from the gages GDB.")
else:
    print("HUC_12 was not present in the gages GDB.")

gdb = None

print("REMAINING GAGES GDB LAYERS:")
for layer in pyogrio.list_layers(gages_gdb):
    print(layer[0])

Successfully deleted HUC_12 from the gages GDB.

REMAINING GAGES GDB LAYERS:
Alabama_Gages_1900
Alabama_Gages_1910
Alabama_Gages_1920
Alabama_Gages_1930
Alabama_Gages_1940
Alabama_Gages_1950
Alabama_Gages_1960
Alabama_Gages_1970
Alabama_Gages_1980
Alabama_Gages_1990
Alabama_Gages_2000
Alabama_Gages_2010
Alabama_Gages_2020


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/osgeo/gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


### Adding state and county boundaries for each state within the ^^ just created processing/Watershed_By_State
### and update crs to nad 83 with state specific utms

In [19]:


project_folder = Path(
    "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research"
)

watershed_folder = (
    project_folder / "data" / "processed" / "Watersheds_By_State"
)


states = {
    "Alabama":     {"fips": "01", "epsg": 26916},
    "Arkansas":    {"fips": "05", "epsg": 26915},
    "Iowa":        {"fips": "19", "epsg": 26915},
    "Kentucky":    {"fips": "21", "epsg": 26916},
    "Mississippi": {"fips": "28", "epsg": 26915},
    "Missouri":    {"fips": "29", "epsg": 26915},
    "Nebraska":    {"fips": "31", "epsg": 26914},
    "Oklahoma":    {"fips": "40", "epsg": 26914},
    "Tennessee":   {"fips": "47", "epsg": 26916},
    "West Virginia": {"fips": "54", "epsg": 26917},
}

required_layers = [
    "HUC_8",
    "HUC_10",
    "HUC_12",
    "State_Boundary",
    "Counties"
]


def extract_polygons(geom):
    if geom is None or geom.is_empty:
        return []

    geom = make_valid(geom)

    if geom.geom_type == "Polygon":
        return [geom]

    if geom.geom_type == "MultiPolygon":
        return list(geom.geoms)

    if hasattr(geom, "geoms"):
        parts = []
        for part in geom.geoms:
            parts.extend(extract_polygons(part))
        return parts

    return []


def to_multipolygon(geom):
    parts = extract_polygons(geom)

    if not parts:
        return None

    merged = unary_union(parts)

    if merged.geom_type == "Polygon":
        return MultiPolygon([merged])

    if merged.geom_type == "MultiPolygon":
        return merged

    parts = extract_polygons(merged)

    if not parts:
        return None

    return MultiPolygon(parts)



results = []

for state_name, info in states.items():

    print("\n" + "=" * 65)
    print(f"PROCESSING: {state_name.upper()}")
    print("=" * 65)

    fips = info["fips"]
    target_epsg = info["epsg"]
    target_crs = f"EPSG:{target_epsg}"

    gdb_path = watershed_folder / f"{state_name}_Watersheds.gdb"

    temp_gdb = watershed_folder / (
        f"{state_name}_Watersheds_TEMP.gdb"
    )

    backup_gdb = watershed_folder / (
        f"{state_name}_Watersheds_BACKUP.gdb"
    )

    if not gdb_path.exists():
        print(f"SKIPPED: GDB not found: {gdb_path}")
        results.append({
            "State": state_name,
            "EPSG": target_epsg,
            "Status": "Missing GDB"
        })
        continue


    existing_layers = {
        layer[0]
        for layer in pyogrio.list_layers(gdb_path)
    }

    missing_hucs = [
        layer for layer in ["HUC_8", "HUC_10", "HUC_12"]
        if layer not in existing_layers
    ]

    if missing_hucs:
        raise ValueError(
            f"{state_name} is missing watershed layers: {missing_hucs}"
        )

    layers_to_write = {}

    for layer_name in ["HUC_8", "HUC_10", "HUC_12"]:
        print(f"Reading {layer_name}...")

        layers_to_write[layer_name] = gpd.read_file(
            gdb_path,
            layer=layer_name
        )

    state_boundary = state_data.loc[
        state_data["STATEFP"] == fips
    ].copy()

    if state_boundary.empty:
        raise ValueError(f"No TIGER state boundary found for {state_name}")

    layers_to_write["State_Boundary"] = state_boundary


    state_counties = county_data.loc[
        county_data["STATEFP"] == fips
    ].copy()

    if state_counties.empty:
        raise ValueError(f"No TIGER counties found for {state_name}")

    layers_to_write["Counties"] = state_counties

    print(f"Counties selected: {len(state_counties)}")
    print(f"Target CRS: {target_crs}")

    for layer_name, gdf in layers_to_write.items():

        print(f"Projecting {layer_name}...")

        if gdf.crs is None:
            raise ValueError(
                f"{state_name} {layer_name} has no defined CRS."
            )

        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf.geometry.apply(to_multipolygon)

        gdf = gdf.loc[
            gdf.geometry.notna() & ~gdf.geometry.is_empty
        ].copy()

        if gdf.empty:
            raise ValueError(
                f"{state_name} {layer_name} has no valid geometries."
            )

        gdf = gdf.set_geometry("geometry")

        if gdf.crs.to_epsg() != target_epsg:
            raise ValueError(
                f"CRS mismatch in {state_name} {layer_name}"
            )

        layers_to_write[layer_name] = gdf

        print(
            f"  Features: {len(gdf):,} | "
            f"CRS: {gdf.crs.to_string()}"
        )

    if temp_gdb.exists():
        shutil.rmtree(temp_gdb)

    if backup_gdb.exists():
        raise FileExistsError(
            f"Backup already exists: {backup_gdb}. "
            "Inspect it before proceeding."
        )

    print("Writing temporary geodatabase...")

    for layer_name in required_layers:

        pyogrio.write_dataframe(
            layers_to_write[layer_name],
            temp_gdb,
            layer=layer_name,
            driver="OpenFileGDB",
            geometry_type="MultiPolygon"
        )

        print(f"  Written: {layer_name}")


    written_layers = {
        layer[0]
        for layer in pyogrio.list_layers(temp_gdb)
    }

    if not set(required_layers).issubset(written_layers):
        raise RuntimeError(
            f"Verification failed for {state_name}: "
            f"{written_layers}"
        )

    for layer_name in required_layers:

        check = gpd.read_file(
            temp_gdb,
            layer=layer_name
        )

        if check.crs.to_epsg() != target_epsg:
            raise RuntimeError(
                f"Incorrect CRS in {state_name} {layer_name}"
            )

        if check.empty:
            raise RuntimeError(
                f"{state_name} {layer_name} is empty."
            )

 
    print("All five layers verified.")

    gdb_path.rename(backup_gdb)

    try:
        temp_gdb.rename(gdb_path)
    except Exception:
        backup_gdb.rename(gdb_path)
        raise

    # Remove backup after successful replacement
    shutil.rmtree(backup_gdb)

    print(f"\nSUCCESS: {state_name}")
    print(f"Geodatabase: {gdb_path}")
    print(f"CRS: {target_crs}")

    results.append({
        "State": state_name,
        "EPSG": target_epsg,
        "CRS": target_crs,
        "HUC_8": len(layers_to_write["HUC_8"]),
        "HUC_10": len(layers_to_write["HUC_10"]),
        "HUC_12": len(layers_to_write["HUC_12"]),
        "Counties": len(layers_to_write["Counties"]),
        "Status": "Successfully updated"
    })



summary = pd.DataFrame(results)

print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)

display(summary)

summary_path = (
    watershed_folder / "Watershed_GDB_UTM_Summary.csv"
)

summary.to_csv(summary_path, index=False)

print(f"Summary saved to: {summary_path}")



PROCESSING: ALABAMA
Reading HUC_8...
Reading HUC_10...
Reading HUC_12...
Counties selected: 67
Target CRS: EPSG:26916
Projecting HUC_8...
  Features: 53 | CRS: EPSG:26916
Projecting HUC_10...
  Features: 321 | CRS: EPSG:26916
Projecting HUC_12...
  Features: 1,487 | CRS: EPSG:26916
Projecting State_Boundary...
  Features: 1 | CRS: EPSG:26916
Projecting Counties...
  Features: 67 | CRS: EPSG:26916

Writing temporary geodatabase...
  Written: HUC_8
  Written: HUC_10
  Written: HUC_12
  Written: State_Boundary
  Written: Counties


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION


All five layers verified.

SUCCESS: Alabama
Geodatabase: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Watersheds_By_State/Alabama_Watersheds.gdb
CRS: EPSG:26916

PROCESSING: ARKANSAS
Reading HUC_8...
Reading HUC_10...
Reading HUC_12...
Counties selected: 75
Target CRS: EPSG:26915
Projecting HUC_8...
  Features: 58 | CRS: EPSG:26915
Projecting HUC_10...
  Features: 308 | CRS: EPSG:26915
Projecting HUC_12...
  Features: 1,556 | CRS: EPSG:26915
Projecting State_Boundary...
  Features: 1 | CRS: EPSG:26915
Projecting Counties...
  Features: 75 | CRS: EPSG:26915

Writing temporary geodatabase...
  Written: HUC_8
  Written: HUC_10
  Written: HUC_12
  Written: State_Boundary


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION

  Written: Counties

All five layers verified.

SUCCESS: Arkansas
Geodatabase: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Watersheds_By_State/Arkansas_Watersheds.gdb
CRS: EPSG:26915

PROCESSING: IOWA
Reading HUC_8...
Reading HUC_10...
Reading HUC_12...
Counties selected: 99
Target CRS: EPSG:26915
Projecting HUC_8...
  Features: 56 | CRS: EPSG:26915
Projecting HUC_10...
  Features: 389 | CRS: EPSG:26915
Projecting HUC_12...
  Features: 1,713 | CRS: EPSG:26915
Projecting State_Boundary...
  Features: 1 | CRS: EPSG:26915
Projecting Counties...
  Features: 99 | CRS: EPSG:26915

Writing temporary geodatabase...
  Written: HUC_8
  Written: HUC_10
  Written: HUC_12
  Written: State_Boundary
  Written: Counties

All five layers verified.

SUCCESS: Iowa
Geodatabase: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Watersheds_By_State/Iowa_Watersheds.gdb
CRS: EPSG:26915

PROCESSING: KENTUCKY
Reading HUC_8...


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION

... (10 outputs left out)

Reading HUC_10...
Reading HUC_12...
Counties selected: 95
Target CRS: EPSG:26916
Projecting HUC_8...
  Features: 60 | CRS: EPSG:26916
Projecting HUC_10...
  Features: 273 | CRS: EPSG:26916
Projecting HUC_12...
  Features: 1,152 | CRS: EPSG:26916
Projecting State_Boundary...
  Features: 1 | CRS: EPSG:26916
Projecting Counties...
  Features: 95 | CRS: EPSG:26916

Writing temporary geodatabase...
  Written: HUC_8
  Written: HUC_10
  Written: HUC_12
  Written: State_Boundary
  Written: Counties

All five layers verified.

SUCCESS: Tennessee
Geodatabase: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Watersheds_By_State/Tennessee_Watersheds.gdb
CRS: EPSG:26916

PROCESSING: WEST VIRGINIA
SKIPPED: GDB not found: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Watersheds_By_State/West Virginia_Watersheds.gdb

FINAL SUMMARY


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION

,State,EPSG,CRS,HUC_8,HUC_10,HUC_12,Counties,Status
0,Alabama,26916,EPSG:26916,53.0,321.0,1487.0,67.0,Successfully updated
1,Arkansas,26915,EPSG:26915,58.0,308.0,1556.0,75.0,Successfully updated
2,Iowa,26915,EPSG:26915,56.0,389.0,1713.0,99.0,Successfully updated
3,Kentucky,26916,EPSG:26916,46.0,249.0,1302.0,120.0,Successfully updated
4,Mississippi,26915,EPSG:26915,55.0,327.0,1469.0,82.0,Successfully updated
5,Missouri,26915,EPSG:26915,67.0,388.0,2000.0,115.0,Successfully updated
6,Nebraska,26914,EPSG:26914,72.0,407.0,2105.0,93.0,Successfully updated
7,Oklahoma,26914,EPSG:26914,70.0,327.0,2079.0,77.0,Successfully updated
8,Tennessee,26916,EPSG:26916,60.0,273.0,1152.0,95.0,Successfully updated
9,West Virginia,26917,NaN,NaN,NaN,NaN,NaN,Missing GDB



Summary saved to: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Watersheds_By_State/Watershed_GDB_UTM_Summary.csv


## Fixing and adding west virginia

In [22]:
state_name = "West Virginia"
fips = "54"
target_epsg = 26917
target_crs = "EPSG:26917"

gdb_path = watershed_folder / "West_Virginia_Watersheds.gdb"
temp_gdb = watershed_folder / "West_Virginia_Watersheds_TEMP.gdb"
backup_gdb = watershed_folder / "West_Virginia_Watersheds_BACKUP.gdb"

required_layers = ["HUC_8", "HUC_10", "HUC_12", "State_Boundary", "Counties"]

layers_to_write = {}

for layer_name in ["HUC_8", "HUC_10", "HUC_12"]:
    layers_to_write[layer_name] = gpd.read_file(
        gdb_path, layer=layer_name
    )

layers_to_write["State_Boundary"] = state_data.loc[
    state_data["STATEFP"] == fips
].copy()

layers_to_write["Counties"] = county_data.loc[
    county_data["STATEFP"] == fips
].copy()

for layer_name, gdf in layers_to_write.items():
    gdf = gdf.to_crs(target_crs)
    gdf["geometry"] = gdf.geometry.apply(to_multipolygon)
    gdf = gdf.loc[
        gdf.geometry.notna() & ~gdf.geometry.is_empty
    ].copy()

    layers_to_write[layer_name] = gdf

    print(f"{layer_name}: {len(gdf):,} features | {gdf.crs}")

if temp_gdb.exists():
    shutil.rmtree(temp_gdb)

if backup_gdb.exists():
    raise FileExistsError(
        f"Backup already exists: {backup_gdb}"
    )

for layer_name in required_layers:
    pyogrio.write_dataframe(
        layers_to_write[layer_name],
        temp_gdb,
        layer=layer_name,
        driver="OpenFileGDB",
        geometry_type="MultiPolygon"
    )

for layer_name in required_layers:
    check = gpd.read_file(temp_gdb, layer=layer_name)

    assert not check.empty, f"{layer_name} is empty"
    assert check.crs.to_epsg() == target_epsg, (
        f"Incorrect CRS in {layer_name}"
    )

gdb_path.rename(backup_gdb)

try:
    temp_gdb.rename(gdb_path)
except Exception:
    backup_gdb.rename(gdb_path)
    raise

shutil.rmtree(backup_gdb)

print("SUCCESS: West Virginia updated.")
print(f"Geodatabase: {gdb_path}")
print(f"CRS: {target_crs}")

HUC_8: 37 features | EPSG:26917
HUC_10: 175 features | EPSG:26917
HUC_12: 785 features | EPSG:26917
State_Boundary: 1 features | EPSG:26917
Counties: 55 features | EPSG:26917
SUCCESS: West Virginia updated.
Geodatabase: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/Watersheds_By_State/West_Virginia_Watersheds.gdb
CRS: EPSG:26917


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field ALAND of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field AWATER of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION

### Now have GDBS for each of our 10 states including state and county boundaries as well as watershed hcu 12, 10, and 8 here /processed/Watersheds_PoliticalBoundaries_By_State